In [2]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Load MNIST
X, y = fetch_openml('mnist_784', version=1, return_X_y=True)
X = X.to_numpy().reshape(-1, 28, 28) / 255.0
y = y.to_numpy().astype(int)

# Use small subset (fast training)
X = X[:1000]
y = y[:1000]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Initialize parameters
num_filters = 2
filter_size = 3
filters = np.random.randn(num_filters, filter_size, filter_size) / 9

fc_weights = np.random.randn(13*13*2, 10) / 100
learning_rate = 0.01

# Convolution
def convolve(image, filters):
    h, w = image.shape
    f = filters.shape[1]
    output = np.zeros((h - f + 1, w - f + 1, filters.shape[0]))

    for k in range(filters.shape[0]):
        for i in range(h - f + 1):
            for j in range(w - f + 1):
                region = image[i:i+f, j:j+f]
                output[i, j, k] = np.sum(region * filters[k])
    return output

# ReLU
def relu(x):
    return np.maximum(0, x)

# Max Pooling
def maxpool(x):
    h, w, d = x.shape
    output = np.zeros((h//2, w//2, d))

    for k in range(d):
        for i in range(0, h, 2):
            for j in range(0, w, 2):
                output[i//2, j//2, k] = np.max(x[i:i+2, j:j+2, k])
    return output

# Softmax
def softmax(x):
    exp = np.exp(x - np.max(x))
    return exp / np.sum(exp)

# Forward + Backward
def train_step(image, label):
    global filters, fc_weights

    # Forward
    conv = convolve(image, filters)
    relu_out = relu(conv)
    pool = maxpool(relu_out)
    flat = pool.flatten()

    logits = np.dot(flat, fc_weights)
    probs = softmax(logits)

    # Loss gradient
    dL = probs
    dL[label] -= 1

    # Backprop (FC layer)
    dW_fc = np.outer(flat, dL)

    # Update weights
    fc_weights -= learning_rate * dW_fc

    return -np.log(probs[label])

# Training loop
for epoch in range(2):  # keep small
    loss = 0
    for i in range(200):  # small batch
        loss += train_step(X_train[i], y_train[i])

    print(f"Epoch {epoch+1}, Loss: {loss/200:.4f}")

# Testing
correct = 0
for i in range(100):
    conv = convolve(X_test[i], filters)
    relu_out = relu(conv)
    pool = maxpool(relu_out)
    flat = pool.flatten()

    logits = np.dot(flat, fc_weights)
    pred = np.argmax(logits)

    if pred == y_test[i]:
        correct += 1

print("Test Accuracy:", correct / 100)

/tmp/ipykernel_7271/2550733871.py:81: RuntimeWarning: invalid value encountered in log
  return -np.log(probs[label])


Epoch 1, Loss: nan
Epoch 2, Loss: nan
Test Accuracy: 0.57
